<a href="https://colab.research.google.com/github/veerakumar17/veera-codeboosters-2026/blob/main/day4/day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
!pip install pyspark
print('pyspark installation complete!')

pyspark installation complete!


In [46]:
#import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as functions
from pyspark.sql.functions import year, month,to_date,col, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName('Day4_BigData_Sales')\
        .config('spark.sql.adaptive.enabled', 'true')\
            .getOrCreate()

print(f'Spark Version: {spark.version}')
print(f'SparkSession: Active')
print(f'application :{spark.sparkContext.appName}')

Spark Version: 4.0.2
SparkSession: Active
application :Day4_BigData_Sales


In [47]:
df_bronze = spark.read\
     .option('header', 'true' )\
          .option('inferSchema', 'true')\
               .csv('large_sales_data.csv')
print('=== BRONZE LAYER - Raw Data ===')
print(f'rows : {df_bronze.count()}')
print(f'columns : {len(df_bronze.columns)}')
print(f'Names : {df_bronze.columns}')
print()
df_bronze.printSchema()

=== BRONZE LAYER - Raw Data ===
rows : 5000
columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [48]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity', 'unit_price', 'revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [49]:
df_bronze.write\
      .mode('overwrite') \
      .parquet('sales_bronze_parquet')

print('bronze parquet saved: sales_bronze.parquet')

import os

def get_dir_size(path):
    """get total size of a file or directory in kb"""
    if os.path.isfile(path):
      return os.path.getsize(path) / 1024
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
      for f in filenames:
        total += os.path.getsize(os.path.join(dirpath, f)) / 1024
    return total

csv_size = get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze_parquet')
reduction = (1- parquet_size/csv_size)*100

print(f'csv size: {csv_size:.2f} kb')
print(f'parquet size: {parquet_size:.2f} kb')
print(f'reduction: {reduction:.2f}%')
print(f'\nAt 1 tb scale: csv=1000 gb -> parquet={1000*(1-reduction/100):.0f}gb')


bronze parquet saved: sales_bronze.parquet
csv size: 529.31 kb
parquet size: 55.10 kb
reduction: 89.59%

At 1 tb scale: csv=1000 gb -> parquet=104gb


In [50]:
df_bronze = df_bronze.withColumn('total_price_and_revenue', col('unit_price') + col('revenue'))
df_bronze.show(5, truncate=False)

df_bronze.printSchema()

+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-----------------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|total_price_and_revenue|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-----------------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |286000                 |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |132000                 |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |8800 

In [51]:
df_bronze.select('quantity', 'unit_price', 'revenue', 'total_price_and_revenue').show(5)
df_bronze.filter(functions.col("revenue") > 50000).show()

+--------+----------+-------+-----------------------+
|quantity|unit_price|revenue|total_price_and_revenue|
+--------+----------+-------+-----------------------+
|      12|     22000| 264000|                 286000|
|      10|     12000| 120000|                 132000|
|      10|       800|   8000|                   8800|
|       5|     32000| 160000|                 192000|
|       4|      3500|  14000|                  17500|
+--------+----------+-------+-----------------------+
only showing top 5 rows
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-----------------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|total_price_and_revenue|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-----------

In [52]:
df_bronze.groupby("category") \
  .agg(functions.sum("revenue").alias("total_revenue")) \
    .orderBy(functions.col("total_revenue"))

DataFrame[category: string, total_revenue: bigint]

In [53]:
df_silver=df_bronze \
  .dropDuplicates() \
    .dropna(subset=['order_id','product','revenue'])

df_silver = df_silver \
  .withColumn('order_date',to_date(col('order_date'),'dd-MM-yyyy')) \
    .withColumn('order_year',year(col('order_date'))) \
      .withColumn('order_month',month(col('order_date')))

df_silver=df_silver.withColumn('revenue_category',
                                 functions.when(col('revenue')>40000,'High')
                                 .when(col('revenue')>10000,'Medium')
                                 .otherwise('Low'))
print(f'Silver layer rows:{df_silver.count()}')
print("New columns added: order_date, order_year, order_month, revenue_category")
df_silver.select('order_date','order_year','order_month','revenue_category').show(5)
#df_silver.show(5,truncate=False)

Silver layer rows:5000
New columns added: order_date, order_year, order_month, revenue_category
+----------+----------+-----------+----------------+
|order_date|order_year|order_month|revenue_category|
+----------+----------+-----------+----------------+
|2023-11-08|      2023|         11|            High|
|2023-01-22|      2023|          1|          Medium|
|2023-09-29|      2023|          9|            High|
|2023-08-12|      2023|          8|            High|
|2023-11-09|      2023|         11|            High|
+----------+----------+-----------+----------------+
only showing top 5 rows


In [54]:
df_silver.write \
   .mode('overwrite') \
      .parquet('dales_silver.parquet')
print('silver Parquet save: sales_silver.parquet')
print(f'Silver size: {get_dir_size("dales_silver.parquet"):.1f}KB')

df_verify = spark.read.parquet('dales_silver.parquet')
print(f'Read-back rows: {df_verify.count()} (should match silver count)')
df_verify.printSchema()

silver Parquet save: sales_silver.parquet
Silver size: 64.9KB
Read-back rows: 5000 (should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total_price_and_revenue: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [55]:
top_product = df_silver \
     .groupBy('product')\
          .agg(\
                   functions.sum('revenue').alias('total_revenue'),\
                            functions.count('order_id').alias('num_order'),\
                                     functions.avg('revenue').alias('avg_order_revenue')\
                                          )\
                                               .orderBy('total_revenue', ascending=False)\
                                                    .limit(5)
print('=== Top 5 products by revenue ===')
top_product.show(truncate=False)

=== Top 5 products by revenue ===
+-------+-------------+---------+------------------+
|product|total_revenue|num_order|avg_order_revenue |
+-------+-------------+---------+------------------+
|Laptop |182700000    |502      |363944.22310756973|
|Tablet |135104000    |532      |253954.8872180451 |
|Monitor|82126000     |481      |170740.12474012474|
|Printer|44544000     |488      |91278.68852459016 |
|Speaker|16317000     |470      |34717.02127659575 |
+-------+-------------+---------+------------------+



In [56]:
# query 1 : top 5 products by total revenue

top_products=df_silver \
  .groupby('product') \
    .agg(\
          functions.sum('revenue').alias('total_revenue'),\
                functions.count('order_id').alias('num_order'),\
                      spark_round(functions.avg('revenue'), 2).alias('avg_revenue')\
                        ) \
                          .orderBy(functions.col('total_revenue').asc()) \
                            .limit(10)
top_products.show(truncate=False)

+----------+-------------+---------+-----------+
|product   |total_revenue|num_order|avg_revenue|
+----------+-------------+---------+-----------+
|USB Hub   |2447400      |527      |4644.02    |
|Mouse     |3207200      |492      |6518.7     |
|Keyboard  |4878000      |495      |9854.55    |
|Webcam    |10982500     |532      |20643.8    |
|Headphones|13541500     |481      |28152.81   |
|Speaker   |16317000     |470      |34717.02   |
|Printer   |44544000     |488      |91278.69   |
|Monitor   |82126000     |481      |170740.12  |
|Tablet    |135104000    |532      |253954.89  |
|Laptop    |182700000    |502      |363944.22  |
+----------+-------------+---------+-----------+



In [57]:
monthly_revenue_trend=df_silver \
  .groupby('order_month') \
    .agg(\
          functions.sum('revenue').alias('monthly_revenue'),\
                functions.count('order_id').alias('monthly_order')\
                  ) \
                    .orderBy(functions.col('order_month').asc()) \
                      .limit(12)
top_products.show(truncate=False)

+----------+-------------+---------+-----------+
|product   |total_revenue|num_order|avg_revenue|
+----------+-------------+---------+-----------+
|USB Hub   |2447400      |527      |4644.02    |
|Mouse     |3207200      |492      |6518.7     |
|Keyboard  |4878000      |495      |9854.55    |
|Webcam    |10982500     |532      |20643.8    |
|Headphones|13541500     |481      |28152.81   |
|Speaker   |16317000     |470      |34717.02   |
|Printer   |44544000     |488      |91278.69   |
|Monitor   |82126000     |481      |170740.12  |
|Tablet    |135104000    |532      |253954.89  |
|Laptop    |182700000    |502      |363944.22  |
+----------+-------------+---------+-----------+



In [58]:
monthly_revenue_trend=df_silver \
  .groupby('order_month') \
    .agg(\
          functions.sum('revenue').alias('monthly_revenue'),\
                functions.count('order_id').alias('monthly_order')\
                  ) \
                    .orderBy(functions.col('order_month').asc()) \
                      .limit(12)

# Add a column for month names
monthly_revenue_trend = monthly_revenue_trend.withColumn(
    'order_month_name',
        functions.date_format(functions.to_date(functions.concat(functions.lit('2023-'), functions.lpad(functions.col('order_month'), 2, '0'), functions.lit('-01'))), 'MMM')
        )

print('=== Monthly Revenue Trend ===')
monthly_revenue_trend.select('order_month_name', 'monthly_revenue', 'monthly_order').show(truncate=False)

=== Monthly Revenue Trend ===
+----------------+---------------+-------------+
|order_month_name|monthly_revenue|monthly_order|
+----------------+---------------+-------------+
|Jan             |41068200       |423          |
|Feb             |34485400       |375          |
|Mar             |40031200       |451          |
|Apr             |38857100       |390          |
|May             |39984500       |423          |
|Jun             |40707400       |390          |
|Jul             |42640700       |405          |
|Aug             |43718500       |418          |
|Sep             |37640200       |398          |
|Oct             |47839000       |479          |
|Nov             |44577100       |419          |
|Dec             |44298300       |429          |
+----------------+---------------+-------------+

